# SurfaceTarget-CRC — phase 05 — patch 5 4

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.4
#
# INTEGRATED CANDIDATE PRIORITIZATION
# AND RANKING SENSITIVITY ANALYSIS
#
# OBJECTIVES:
# 1. Validate Patch 5.3 evidence triage
# 2. Preserve all 948 research candidates
# 3. Construct quantitative RNA evidence measures
# 4. Preserve qualifying CPTAC evidence
# 5. Preserve normal-tissue expression concerns
# 6. Separate biological targeting categories
# 7. Calculate three transparent priority scenarios
# 8. Compare ranking sensitivity
# 9. Identify stable research follow-up candidates
# 10. Review leading genes separately by targeting lane
# 11. Save prioritization tables and provenance
# 12. Create a research checkpoint
#
# IMPORTANT:
# - Research-priority rankings, NOT therapeutic efficacy scores
# - No clinical safety predictions
# - No automatic gene exclusions
# - No single ranking across incompatible target mechanisms
# - Missing HPA evidence is NOT favorable safety evidence
# - Missing CPTAC evidence is NOT absent protein
# - CRISPR dependency is NOT required for ADC targeting
# - No experimentally validated targets are claimed
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np

import hashlib
import json


ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    RESULTS,
    META,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


SOURCE_PATH = (
    RESULTS /
    "surfacetarget_phase5_evidence_triage_v1.csv"
)

PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_evidence_triage_provenance.json"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


SURFACE_LANE = (
    "Putative cell-surface antigen review"
)

STROMAL_LANE = (
    "Stromal-associated targeting review"
)

MEMBRANE_REVIEW_LANE = (
    "Membrane localization/accessibility verification"
)

EXTRACELLULAR_LANE = (
    "Extracellular microenvironment review"
)

ALTERNATIVE_LANE = (
    "Topology clarification or alternative modality review"
)


EXPECTED_LANES = {
    SURFACE_LANE: 111,
    STROMAL_LANE: 1,
    MEMBRANE_REVIEW_LANE: 152,
    EXTRACELLULAR_LANE: 128,
    ALTERNATIVE_LANE: 556
}


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.4")
print("INTEGRATED CANDIDATE PRIORITIZATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


for path in [
    SOURCE_PATH,
    PROVENANCE_PATH
]:

    if not path.exists():

        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    PROVENANCE_PATH,
    "r"
) as handle:

    previous_provenance = json.load(handle)


actual_hash = sha256_file(SOURCE_PATH)

expected_hash = previous_provenance.get(
    "output_sha256"
)


if not expected_hash:

    raise RuntimeError(
        "Patch 5.3 provenance has no output checksum."
    )


if actual_hash != expected_hash:

    raise RuntimeError(
        "Patch 5.3 triage table checksum mismatch."
    )


print("\nSource checksum: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD AND VALIDATE RESEARCH CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING EVIDENCE TRIAGE DATA")
print("-" * 65)


source = pd.read_csv(
    SOURCE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


required = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "phase5_followup_lane",
    "phase5_topology_class",
    "phase5_protein_evidence_state",
    "phase5_normal_tissue_review_state",
    "phase5_rna_evidence_pattern",
    "phase5_stronger_surface_annotation",
    "protein_enriched",
    "baseline_log2fc",
    "sensitivity_log2fc",
    "tier_a_median_cpm",
    "tier_a_fraction_cpm_ge1",
    "mss_complete_pairs",
    "phase5_normal_ihc_available",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect"
}


missing = required - set(source.columns)

if missing:

    raise RuntimeError(
        f"Required evidence columns missing: "
        f"{sorted(missing)}"
    )


if len(source) != 948:

    raise RuntimeError(
        "Expected 948 research candidates."
    )


if source["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene IDs."
    )


if source["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene symbols."
    )


print("\nCandidate genes:", len(source))

print("Evidence fields:", len(source.columns))

print("Candidate identity: PASSED")


# ------------------------------------------------------------
# 4. VALIDATE BIOLOGICAL FOLLOW-UP LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING TARGETING CATEGORIES")
print("-" * 65)


observed_lanes = (
    source["phase5_followup_lane"]
    .value_counts()
    .to_dict()
)


if observed_lanes != EXPECTED_LANES:

    raise RuntimeError(
        "Biological follow-up categories differ "
        "from the validated Patch 5.3 results."
    )


print("\nFollow-up lane distribution:")

print(
    source["phase5_followup_lane"]
    .value_counts()
    .to_string()
)


print("\nBiological lane validation: PASSED")


# ------------------------------------------------------------
# 5. PREPARE ANALYSIS HELPERS
# ------------------------------------------------------------

def as_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    allowed = {
        "true",
        "false",
        "1",
        "0",
        "",
        "nan",
        "none",
        "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(allowed))
    )

    if unexpected.any():

        raise RuntimeError(
            "Unexpected boolean annotation: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


def numeric(column):

    return pd.to_numeric(
        source[column],
        errors="coerce"
    )


data = source.copy(deep=True)


# ------------------------------------------------------------
# 6. QUANTITATIVE RNA EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING TRANSCRIPTOMIC PRIORITIZATION EVIDENCE")
print("-" * 65)


baseline = numeric(
    "baseline_log2fc"
)

sensitivity = numeric(
    "sensitivity_log2fc"
)

tier_a_cpm = numeric(
    "tier_a_median_cpm"
)

tier_a_fraction = numeric(
    "tier_a_fraction_cpm_ge1"
)


if baseline.isna().any():

    raise RuntimeError(
        "Missing baseline RNA effect sizes."
    )


if sensitivity.isna().any():

    raise RuntimeError(
        "Missing sensitivity RNA effect sizes."
    )


if tier_a_fraction.isna().any():

    raise RuntimeError(
        "Missing Tier A RNA detection fractions."
    )


if not tier_a_fraction.between(
    0,
    1
).all():

    raise RuntimeError(
        "Tier A expression fractions outside [0,1]."
    )


# Conservative descriptive RNA magnitude:
#
# Use the smaller of the baseline and
# QC-screened log2 fold changes.
#
# This prevents a candidate from benefiting
# exclusively from a larger estimate in
# just one of the two related analyses.
#
# This is NOT an independent validation score.


data["phase5_rna_conservative_log2fc"] = (
    np.minimum(
        baseline.to_numpy(dtype=float),
        sensitivity.to_numpy(dtype=float)
    )
)


data["phase5_rna_tier_a_detection_fraction"] = (
    tier_a_fraction
)


data["phase5_rna_tier_a_median_cpm"] = (
    tier_a_cpm
)


rna_patterns = (
    data["phase5_rna_evidence_pattern"]
    .value_counts()
)


print("\nRNA evidence pattern distribution:")

print(
    rna_patterns.to_string()
)


if len(rna_patterns) == 1:

    print(
        "\nAll candidates share the same RNA "
        "evidence category."
    )

    print(
        "Continuous effect sizes, rather than "
        "the shared eligibility category, "
        "will distinguish candidates."
    )


print(
    "\nConservative log2FC range:",
    round(
        data[
            "phase5_rna_conservative_log2fc"
        ].min(),
        3
    ),
    "to",
    round(
        data[
            "phase5_rna_conservative_log2fc"
        ].max(),
        3
    )
)


# ------------------------------------------------------------
# 7. PREPARE QUALIFYING PROTEIN EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CPTAC PROTEIN EVIDENCE")
print("-" * 65)


protein_supported = as_bool(
    data["protein_enriched"]
)


if int(protein_supported.sum()) != 20:

    raise RuntimeError(
        "Expected 20 qualifying protein-enrichment genes."
    )


data["phase5_qualifying_protein_evidence"] = (
    protein_supported
)


# A qualifying enrichment result is rewarded
# as positive evidence.
#
# Missing protein measurements are NOT
# assigned a numerical penalty.
#
# The difference between inadequate coverage
# and adequately measured nonqualifying
# results remains visible in the source field.


data["phase5_protein_evidence_label"] = np.where(

    protein_supported,

    "Qualifying CPTAC enrichment observed",

    "Qualifying CPTAC enrichment not established"
)


print("\nQualifying protein-enrichment evidence:",
      int(protein_supported.sum()))


print(
    "Other candidates:",
    int((~protein_supported).sum())
)


print(
    "\nUnestablished enrichment is not "
    "interpreted as absent protein."
)


# ------------------------------------------------------------
# 8. PREPARE NORMAL-TISSUE REVIEW CATEGORIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


normal_state_order = {

    (
        "No medium/high focus-organ signal "
        "in available summary; not evidence of safety"
    ): 0,

    (
        "Reliable medium/high staining "
        "reported in other normal tissues"
    ): 1,

    (
        "Medium/high staining in focus organ; "
        "reliability requires review"
    ): 2,

    (
        "Medium/high staining in focus organ "
        "with stronger IHC reliability"
    ): 3,

    "Normal-tissue IHC unavailable": 4

}


unexpected_states = (
    set(data["phase5_normal_tissue_review_state"])
    - set(normal_state_order)
)


if unexpected_states:

    raise RuntimeError(
        "Unrecognized normal-tissue review states: "
        f"{unexpected_states}"
    )


data["phase5_normal_review_order"] = (
    data["phase5_normal_tissue_review_state"]
    .map(normal_state_order)
    .astype(int)
)


normal_available = as_bool(
    data["phase5_normal_ihc_available"]
)


if int(normal_available.sum()) != 690:

    raise RuntimeError(
        "Expected 690 candidates with normal-IHC evidence."
    )


# The ordinal review order is only a
# scenario-specific research follow-up rule.
#
# It is NOT a toxicity score.
#
# Unknown normal-tissue expression is placed
# last in the normal-review-first scenario
# to prevent missing evidence from being
# interpreted as favorable selectivity.


print("\nNormal-tissue category distribution:")

print(
    data["phase5_normal_tissue_review_state"]
    .value_counts()
    .to_string()
)


print(
    "\nUnknown IHC evidence is kept separate "
    "from observed low staining."
)


# ------------------------------------------------------------
# 9. VALIDATE SURFACE-RELATED EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SURFACE-RELATED CANDIDATES")
print("-" * 65)


stronger_surface = as_bool(
    data["phase5_stronger_surface_annotation"]
)


if int(stronger_surface.sum()) != 112:

    raise RuntimeError(
        "Expected 112 stronger surface annotations."
    )


surface_mask = (
    data["phase5_followup_lane"]
    .eq(SURFACE_LANE)
)


stromal_mask = (
    data["phase5_followup_lane"]
    .eq(STROMAL_LANE)
)


if int(surface_mask.sum()) != 111:

    raise RuntimeError(
        "Expected 111 putative cell-surface candidates."
    )


if int(stromal_mask.sum()) != 1:

    raise RuntimeError(
        "Expected one stromal-review candidate."
    )


if not stronger_surface[
    surface_mask | stromal_mask
].all():

    raise RuntimeError(
        "Surface-review lane contains candidates "
        "without the expected topology annotation."
    )


print("\nPutative cell-surface candidates:", 111)

print("Stromal-associated candidates:", 1)

print(
    "Qualifying CPTAC-supported surface candidates:",
    int(
        (
            surface_mask
            & protein_supported
        ).sum()
    )
)


print(
    "Qualifying CPTAC-supported stromal candidates:",
    int(
        (
            stromal_mask
            & protein_supported
        ).sum()
    )
)


# ------------------------------------------------------------
# 10. DEFINE TRANSPARENT PRIORITIZATION SCENARIOS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING PRIORITIZATION SCENARIOS")
print("-" * 65)


# We do not combine the evidence into
# a single weighted numerical score.
#
# Instead, each scenario uses a transparent
# ordering of the same observed variables.
#
# Ranking is performed separately inside
# each biological follow-up lane.


SCENARIOS = {

    "protein_first": {

        "columns": [
            "phase5_qualifying_protein_evidence",
            "phase5_normal_review_order",
            "phase5_rna_conservative_log2fc",
            "phase5_rna_tier_a_detection_fraction",
            "gene_id"
        ],

        "ascending": [
            False,
            True,
            False,
            False,
            True
        ],

        "description": (
            "Qualifying protein enrichment first; "
            "then normal-tissue review status; "
            "then conservative RNA effect size"
        )
    },

    "normal_review_first": {

        "columns": [
            "phase5_normal_review_order",
            "phase5_qualifying_protein_evidence",
            "phase5_rna_conservative_log2fc",
            "phase5_rna_tier_a_detection_fraction",
            "gene_id"
        ],

        "ascending": [
            True,
            False,
            False,
            False,
            True
        ],

        "description": (
            "Normal-tissue expression review first; "
            "then qualifying protein evidence; "
            "then conservative RNA effect size"
        )
    },

    "rna_discovery_first": {

        "columns": [
            "phase5_rna_conservative_log2fc",
            "phase5_rna_tier_a_detection_fraction",
            "phase5_qualifying_protein_evidence",
            "phase5_normal_review_order",
            "gene_id"
        ],

        "ascending": [
            False,
            False,
            False,
            True,
            True
        ],

        "description": (
            "Conservative RNA enrichment first; "
            "then Tier A detection breadth; "
            "then protein and normal-tissue context"
        )
    }

}


for name, settings in SCENARIOS.items():

    print("\nScenario:", name)

    print(settings["description"])


print("\nIMPORTANT:")

print(
    "These are research review priorities, "
    "not clinical target scores."
)

print(
    "The normal-review scenario evaluates "
    "annotation concerns, not actual toxicity."
)


# ------------------------------------------------------------
# 11. CALCULATE WITHIN-LANE SCENARIO RANKS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING BIOLOGICAL LANE-SPECIFIC RANKS")
print("-" * 65)


for scenario_name, settings in SCENARIOS.items():

    ranking_lookup = {}

    for lane, group in data.groupby(
        "phase5_followup_lane",
        sort=True
    ):

        ordered = group.sort_values(

            by=settings["columns"],

            ascending=settings["ascending"],

            na_position="last",

            kind="mergesort"

        )


        for rank, gene_id in enumerate(
            ordered["gene_id"].tolist(),
            start=1
        ):

            if gene_id in ranking_lookup:

                raise RuntimeError(
                    "Gene assigned multiple scenario ranks."
                )

            ranking_lookup[gene_id] = rank


    if len(ranking_lookup) != 948:

        raise RuntimeError(
            "Not all genes received a scenario rank."
        )


    output_column = (
        f"phase5_rank_{scenario_name}"
    )


    data[output_column] = (
        data["gene_id"]
        .map(ranking_lookup)
        .astype(int)
    )


print("\nThree ranking scenarios calculated.")


# ------------------------------------------------------------
# 12. VALIDATE WITHIN-LANE RANK UNIQUENESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SCENARIO RANKINGS")
print("-" * 65)


rank_columns = [
    f"phase5_rank_{name}"
    for name in SCENARIOS
]


for lane, group in data.groupby(
    "phase5_followup_lane"
):

    expected_ranks = set(
        range(1, len(group) + 1)
    )

    for column in rank_columns:

        observed_ranks = set(
            group[column].astype(int)
        )

        if observed_ranks != expected_ranks:

            raise RuntimeError(
                "Invalid within-lane ranks: "
                f"{lane}, {column}"
            )


print("\nWithin-lane ranking uniqueness: PASSED")

print(
    "No cross-lane therapeutic hierarchy "
    "has been imposed."
)


# ------------------------------------------------------------
# 13. SUMMARIZE RANKING SENSITIVITY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING RANKING SENSITIVITY")
print("-" * 65)


data["phase5_median_scenario_rank"] = (
    data[rank_columns]
    .median(axis=1)
)


data["phase5_best_scenario_rank"] = (
    data[rank_columns]
    .min(axis=1)
)


data["phase5_worst_scenario_rank"] = (
    data[rank_columns]
    .max(axis=1)
)


data["phase5_rank_range"] = (

    data["phase5_worst_scenario_rank"]
    -
    data["phase5_best_scenario_rank"]

)


# The top-20 threshold is a convenient
# shortlist size for research review.
#
# It is not a biological threshold.


TOP_N = 20


data["phase5_scenarios_in_top20"] = (
    data[rank_columns]
    .le(TOP_N)
    .sum(axis=1)
)


data["phase5_consistent_top20_review"] = (
    data["phase5_scenarios_in_top20"]
    .ge(2)
)


print(
    "\nCandidates appearing in the top 20 "
    "of at least two scenarios, by lane:"
)


print(

    data.groupby(
        "phase5_followup_lane"
    )["phase5_consistent_top20_review"]

    .sum()

    .to_string()

)


print("\nIMPORTANT:")

print(
    "Median scenario rank summarizes robustness "
    "to three explicit ordering assumptions."
)

print(
    "It is not a therapeutic efficacy "
    "or clinical development score."
)


# ------------------------------------------------------------
# 14. PREPARE THE PRIMARY SURFACE SHORTLIST
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRIMARY CELL-SURFACE CANDIDATE SHORTLIST")
print("-" * 65)


surface = data[
    data["phase5_followup_lane"]
    .eq(SURFACE_LANE)
].copy()


surface = surface.sort_values(

    [
        "phase5_median_scenario_rank",
        "phase5_rank_range",
        "gene_id"
    ],

    ascending=[
        True,
        True,
        True
    ]

)


if len(surface) != 111:

    raise RuntimeError(
        "Surface candidate inventory changed."
    )


surface["phase5_surface_consensus_position"] = (
    np.arange(
        1,
        len(surface) + 1
    )
)


surface_top20 = surface.head(
    TOP_N
).copy()


print("\nTop 20 exploratory surface-review candidates:")


display_columns = [
    "gene_name",
    "phase5_surface_consensus_position",
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first",
    "phase5_scenarios_in_top20",
    "phase5_qualifying_protein_evidence",
    "phase5_normal_tissue_review_state",
    "phase5_rna_conservative_log2fc"
]


print("\n")

print(

    surface_top20[
        display_columns
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )

)


# ------------------------------------------------------------
# 15. REVIEW SHORTLIST STABILITY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRIMARY SURFACE RANKING STABILITY")
print("-" * 65)


scenario_top_sets = {}


for scenario_name in SCENARIOS:

    rank_column = (
        f"phase5_rank_{scenario_name}"
    )

    scenario_top_sets[scenario_name] = set(

        surface.loc[
            surface[rank_column].le(TOP_N),
            "gene_id"
        ]

    )


stability_records = []


scenario_names = list(
    SCENARIOS.keys()
)


for i in range(len(scenario_names)):

    for j in range(i + 1, len(scenario_names)):

        first = scenario_names[i]

        second = scenario_names[j]

        first_set = scenario_top_sets[first]

        second_set = scenario_top_sets[second]

        overlap = len(
            first_set & second_set
        )

        union = len(
            first_set | second_set
        )

        jaccard = (
            overlap / union
            if union > 0
            else np.nan
        )

        stability_records.append({

            "scenario_a": first,

            "scenario_b": second,

            "top_n": TOP_N,

            "shared_candidates": overlap,

            "jaccard_similarity": jaccard

        })


stability = pd.DataFrame(
    stability_records
)


print("\nPairwise top-20 overlap:")

print(
    stability.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


stable_surface = surface[
    surface["phase5_consistent_top20_review"]
].copy()


print(
    "\nSurface candidates in top 20 "
    "under at least two scenarios:",
    len(stable_surface)
)


print("\nStable exploratory follow-up genes:")


if not stable_surface.empty:

    print(

        stable_surface[
            [
                "gene_name",
                "phase5_scenarios_in_top20",
                "phase5_median_scenario_rank",
                "phase5_qualifying_protein_evidence",
                "phase5_normal_tissue_review_state"
            ]
        ]

        .to_string(
            index=False
        )

    )

else:

    print(
        "No genes meet the specified "
        "ranking-consistency criterion."
    )


# ------------------------------------------------------------
# 16. REVIEW STROMAL CANDIDATES SEPARATELY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("STROMAL TARGETING REVIEW")
print("-" * 65)


stromal = data[
    data["phase5_followup_lane"]
    .eq(STROMAL_LANE)
].copy()


if len(stromal) != 1:

    raise RuntimeError(
        "Expected one stromal-associated candidate."
    )


print("\n")

print(
    stromal[
        [
            "gene_name",
            "phase5_protein_evidence_state",
            "phase5_normal_tissue_review_state",
            "phase5_rna_conservative_log2fc",
            "depmap_crc_median_gene_effect"
        ]
    ].to_string(index=False)
)


print(
    "\nStromal and tumor-cell antigen "
    "ranks must not be compared directly."
)


# ------------------------------------------------------------
# 17. REVIEW OTHER BIOLOGICAL LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("OTHER BIOLOGICAL FOLLOW-UP CATEGORIES")
print("-" * 65)


other_lane_records = []


for lane in [
    MEMBRANE_REVIEW_LANE,
    EXTRACELLULAR_LANE,
    ALTERNATIVE_LANE
]:

    subset = data[
        data["phase5_followup_lane"]
        .eq(lane)
    ].copy()

    subset = subset.sort_values(
        [
            "phase5_median_scenario_rank",
            "phase5_rank_range",
            "gene_id"
        ]
    )

    other_lane_records.append(
        subset
    )

    print("\n" + "-" * 50)

    print("Lane:", lane)

    print("Candidates:", len(subset))

    print("\nFirst 10 exploratory review candidates:")

    print(

        subset[
            [
                "gene_name",
                "phase5_median_scenario_rank",
                "phase5_qualifying_protein_evidence",
                "phase5_normal_tissue_review_state"
            ]
        ]

        .head(10)

        .to_string(index=False)

    )


other_lanes = pd.concat(
    other_lane_records,
    ignore_index=True
)


# ------------------------------------------------------------
# 18. REVIEW LEADING BIOLOGICAL CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE SENSITIVITY REVIEW")
print("-" * 65)


leading = data[
    data["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "phase5_followup_lane",
    "phase5_qualifying_protein_evidence",
    "phase5_normal_tissue_review_state",
    "phase5_rna_conservative_log2fc",
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first",
    "phase5_median_scenario_rank"
]


print("\n")

print(

    leading[
        leading_columns
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )

)


print("\nINTERPRETATION:")

print(
    "Ranks are comparable only within "
    "the same biological follow-up lane."
)

print(
    "A candidate with missing normal-IHC "
    "evidence is not assumed to be selective."
)

print(
    "A candidate with weak CRISPR dependency "
    "is not automatically excluded."
)


# ------------------------------------------------------------
# 19. VALIDATE SOURCE EVIDENCE PRESERVATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SOURCE EVIDENCE PRESERVATION")
print("-" * 65)


original_columns = source.columns.tolist()


if not source.equals(
    data[original_columns]
):

    raise RuntimeError(
        "Source evidence changed during prioritization."
    )


if len(data) != 948:

    raise RuntimeError(
        "Candidate count changed."
    )


if data["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate identifiers."
    )


if int(
    data["phase5_qualifying_protein_evidence"]
    .sum()
) != 20:

    raise RuntimeError(
        "Qualifying protein-support count changed."
    )


if int(
    stronger_surface.sum()
) != 112:

    raise RuntimeError(
        "Surface-topology evidence count changed."
    )


print("\nOriginal evidence: UNCHANGED")

print("Candidate preservation: PASSED")

print("Protein support: PASSED")

print("Surface topology: PASSED")


# ------------------------------------------------------------
# 20. PREPARE FINAL REVIEW TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING PRIORITIZATION OUTPUTS")
print("-" * 65)


# Full research table.
#
# These ranks are exploratory and are
# explicitly limited to their own lanes.


ranked = data.copy()


# Add surface consensus position
# without assigning artificial positions
# to candidates in other biological lanes.


surface_positions = dict(
    zip(
        surface["gene_id"],
        surface["phase5_surface_consensus_position"]
    )
)


ranked["phase5_surface_consensus_position"] = (
    ranked["gene_id"]
    .map(surface_positions)
)


if int(
    ranked["phase5_surface_consensus_position"]
    .notna()
    .sum()
) != 111:

    raise RuntimeError(
        "Surface ranking coverage mismatch."
    )


# Scenario methodology table.

scenario_methodology = pd.DataFrame([

    {
        "scenario": name,

        "ordering_columns":
            "; ".join(settings["columns"]),

        "ascending_flags":
            "; ".join(
                str(value)
                for value in settings["ascending"]
            ),

        "interpretation":
            settings["description"]

    }

    for name, settings in SCENARIOS.items()

])


# Research review categories are never
# labeled as safe/unsafe or clinically
# recommended/not recommended.


# ------------------------------------------------------------
# 21. SAVE PRIORITIZATION RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PRIORITIZATION RESULTS")
print("-" * 65)


OUTPUT_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)


ranked.to_csv(
    OUTPUT_PATH,
    index=False
)


surface.to_csv(

    RESULTS /
    "surfacetarget_phase5_surface_ranked_candidates.csv",

    index=False
)


surface_top20.to_csv(

    RESULTS /
    "surfacetarget_phase5_surface_top20_exploratory.csv",

    index=False
)


stable_surface.to_csv(

    RESULTS /
    "surfacetarget_phase5_surface_stable_followup.csv",

    index=False
)


stromal.to_csv(

    RESULTS /
    "surfacetarget_phase5_stromal_target_review.csv",

    index=False
)


other_lanes.to_csv(

    RESULTS /
    "surfacetarget_phase5_alternative_lane_priorities.csv",

    index=False
)


leading.to_csv(

    RESULTS /
    "surfacetarget_phase5_leading_rank_sensitivity.csv",

    index=False
)


stability.to_csv(

    RESULTS /
    "surfacetarget_phase5_ranking_stability.csv",

    index=False
)


scenario_methodology.to_csv(

    RESULTS /
    "surfacetarget_phase5_ranking_scenarios.csv",

    index=False
)


# Reopen and validate the main output.

reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


if len(reloaded) != 948:

    raise RuntimeError(
        "Saved prioritization table lost candidates."
    )


if (
    reloaded["gene_id"].tolist()
    != ranked["gene_id"].tolist()
):

    raise RuntimeError(
        "Saved candidate identifiers changed."
    )


print("\nPrioritization tables: SAVED")

print("Saved table validation: PASSED")


# ------------------------------------------------------------
# 22. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PRIORITIZATION PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.4",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": str(SOURCE_PATH),

    "source_sha256": actual_hash,

    "output": str(OUTPUT_PATH),

    "output_sha256":
        sha256_file(OUTPUT_PATH),

    "candidate_count": 948,

    "surface_lane_candidates": 111,

    "stromal_lane_candidates": 1,

    "stronger_surface_annotations": 112,

    "qualifying_cptac_enrichment_genes": 20,

    "ranking_method": (
        "Three lexicographic research-priority "
        "scenarios within biological follow-up lanes"
    ),

    "scenarios": {

        name: {
            "ordering_columns": settings["columns"],
            "ascending": settings["ascending"],
            "description": settings["description"]
        }

        for name, settings in SCENARIOS.items()

    },

    "rna_metric": (
        "Minimum of baseline and QC-screened "
        "tumor-vs-normal log2 fold changes"
    ),

    "rna_independence_note": (
        "Baseline and QC-screened analyses "
        "are related sensitivity analyses, "
        "not independent experiments"
    ),

    "protein_policy": (
        "Qualifying enrichment treated as "
        "supporting evidence; absence of qualifying "
        "enrichment is not interpreted as absent protein"
    ),

    "normal_tissue_policy": (
        "Missing IHC data assigned a distinct "
        "unassessed category, never a favorable "
        "selectivity classification"
    ),

    "normal_review_order_interpretation": (
        "Scenario-specific annotation-review priority; "
        "not a quantitative clinical toxicity estimate"
    ),

    "dependency_policy": (
        "DepMap CRISPR gene effects retained "
        "as context but not used to determine "
        "antibody-antigen prioritization"
    ),

    "cross_lane_rank_comparison_allowed": False,

    "top_n_for_sensitivity": TOP_N,

    "stable_followup_rule": (
        "Within the same lane, appears in the top "
        "20 under at least two of three scenarios"
    ),

    "candidate_exclusions": 0,

    "therapeutic_efficacy_scores": False,

    "clinical_safety_predictions": False,

    "experimentally_validated_targets": False,

    "limitations": [

        "The research-priority ordering depends "
        "on selected evidence hierarchy assumptions.",

        "A high ranking does not imply a "
        "clinically favorable therapeutic window.",

        "Normal-tissue IHC measurements "
        "are not direct toxicity assessments.",

        "Missing normal-tissue IHC evidence "
        "does not imply low expression.",

        "Bulk tumor RNA and proteomics do "
        "not establish malignant-cell origin.",

        "CPTAC enrichment is exploratory "
        "and based on measured protein coverage.",

        "GPI anchoring and extracellular topology "
        "do not establish antibody accessibility.",

        "CRISPR dependency is not required "
        "for antibody-drug conjugate targeting.",

        "The DepMap colorectal cohort "
        "has not been confirmed MSS.",

        "TCGA Tier B is computationally "
        "low-MSI rather than independently "
        "confirmed MSS.",

        "No target has been experimentally validated."

    ]

}


with open(

    META /
    "surfacetarget_phase5_prioritization_provenance.json",

    "w"

) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance: SAVED")


# ------------------------------------------------------------
# 23. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Phase5_Patch54_{timestamp}.zip"

)


# Preserve analysis results, provenance
# and compact research sources.
#
# Exclude raw TCGA and reconstructed
# DepMap molecular datasets.


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_size_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created.")

print(
    "Size:",
    f"{checkpoint_size_mib:.2f} MiB"
)

print("Location:")
print(checkpoint)


# ------------------------------------------------------------
# 24. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.4 SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(ranked))


print(
    "Primary surface-review candidates:",
    len(surface)
)


print(
    "Stromal-review candidates:",
    len(stromal)
)


print(
    "Qualifying CPTAC-supported genes:",
    int(protein_supported.sum())
)


print(
    "Ranking scenarios:",
    len(SCENARIOS)
)


print(
    "Stable surface follow-up candidates:",
    len(stable_surface)
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_prioritization_v1.csv")

print("2. surfacetarget_phase5_surface_ranked_candidates.csv")

print("3. surfacetarget_phase5_surface_top20_exploratory.csv")

print("4. surfacetarget_phase5_surface_stable_followup.csv")

print("5. surfacetarget_phase5_stromal_target_review.csv")

print("6. surfacetarget_phase5_alternative_lane_priorities.csv")

print("7. surfacetarget_phase5_leading_rank_sensitivity.csv")

print("8. surfacetarget_phase5_ranking_stability.csv")

print("9. surfacetarget_phase5_ranking_scenarios.csv")

print("10. surfacetarget_phase5_prioritization_provenance.json")


print("\nSource evidence: VERIFIED UNCHANGED")

print("Biological lane separation: PRESERVED")

print("Ranking sensitivity: COMPLETE")

print("Research checkpoint: CREATED")


print(
    "Checkpoint size:",
    f"{checkpoint_size_mib:.2f} MiB"
)


print("\nNo candidates excluded.")

print("No clinical safety scores assigned.")

print("No therapeutic efficacy claims.")

print("No experimentally validated targets claimed.")


print("\nPATCH 5.4 COMPLETE")

print("=" * 65)